# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShayanAhmed12/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Lane:** CTR / Engagement Opportunity Scoring (from ML-02/ML-03), now on the warehouse release.

**The contract in five plain answers:**

1. **What one row means:** one **page** (`content_hash_id`, under one `client_hash_id`) at one decision moment. The raw source is the daily fact table, where one row = one page on one day (`report_date × client_hash_id × content_hash_id`). I aggregate it to one row per page.
2. **Which tables:** `fact_content_daily_performance` (partition `month=2026-03`) for search metrics, joined to `dim_content` on `content_hash_id` for slow-moving context (content type, creation date).
3. **Which time window:** a mid-panel month, **March 2026**, split at a decision moment of **2026-03-16**:
   - **feature window:** 2026-03-01 → 2026-03-15 (what an editor could see on decision day)
   - **outcome window:** 2026-03-16 → 2026-03-31 (what happened next)
   - The final month (June 2026, the `_sample` table) is not touched; it stays a sealed test month.
4. **What I predict / rank (proxy label):** `under_captures_next` = 1 if, in the **outcome window**, the page's CTR is below half the median CTR of pages in the same position bucket. It applies only to pages with ≥100 impressions in both windows. The queue later ranks pages by the risk of this happening × impressions.
5. **What I deliberately exclude:** every **outcome-window** metric as a feature (clicks, impressions, CTR or position from 16–31 March). They describe the future the label measures. GA4 engagement columns are also left out for now: they're zero-filled before each client's GA4 start, and my label is about search clicks.

**Data access note:** on this machine DuckDB's remote `hf://` reader would not load, so the cell below downloads the three Parquet files once (~145 MB, cached) with `huggingface_hub` and queries them locally with DuckDB. The token comes from the Hugging Face login cache / `HF_TOKEN` secret, never from a cell.


In [1]:
import os, warnings
warnings.filterwarnings("ignore")
import duckdb, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download, get_token

assert get_token(), "No Hugging Face token: log in (hf auth login) or add HF_TOKEN in Colab Secrets"
REPO = "FlyRank/internship-warehouse"
files = {name: hf_hub_download(REPO, path, repo_type="dataset") for name, path in {
    "fact_mar": "fact_content_daily_performance/month=2026-03/data_0.parquet",
    "dim_content": "dim_content.parquet",
    "dim_clients": "dim_clients.parquet",
}.items()}

con = duckdb.connect()
for name, path in files.items():
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{path}')")

DECISION = "2026-03-16"   # feature window: < DECISION, outcome window: >= DECISION
print("Views ready:", ", ".join(files))


Views ready: fact_mar, dim_content, dim_clients


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** (feature window only, 1–15 Mar) | `impr_f`, `ctr_f`, `pos_f`, `content_type`, `age_days` | knowable on 16 March, see the five lines below |
| **Label / proxy** | `under_captures_next` (from `gsc_clicks`, `gsc_impressions` in 16–31 Mar) | the outcome being predicted, never a feature |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `month` | join, group, split by client; pseudonyms carry no meaning |
| **Excluded** | all 16–31 Mar metrics as features | future information: they are the label in disguise |
| **Excluded** | `ga4_*`, `sessions_*`, `ai_*`, `scroll_events` | zero-filled before GA4 start (`ga4_data_available = FALSE`); not needed for a search-CTR label yet |
| **Excluded** | `keyword_hash_id`, `url_hash_id` | grouping/dedup only, never features; never reverse them |
| **Excluded** | `last_optimized_date`, `optimization_eligible_date` | product-workflow fields, close to the product's own decisions |

**Five features, max, each with "knowable at the decision moment because…":**

1. **`impr_f`** (impressions 1–15 Mar, logged in the model): knowable on 16 Mar because Search Console reports past days' impressions; it only covers days before the decision.
2. **`ctr_f`** (clicks ÷ impressions, 1–15 Mar): knowable because both parts are past Search Console counts from the feature window.
3. **`pos_f`** (impression-weighted average position, 1–15 Mar): knowable because it's computed from feature-window daily positions only, *not* the full-month average, which would overlap the outcome.
4. **`content_type`** (from `dim_content`): knowable because it's set when the content is created and doesn't change with performance.
5. **`age_days`** (16 Mar − `content_created_date`): knowable because the creation date is in the past; age is measured at the decision date, not at export time.


In [2]:
# The field map as data (so it can be checked / reused later)
fields = pd.DataFrame([
    ("impr_f", "feature", "sum(gsc_impressions) where report_date < 2026-03-16"),
    ("ctr_f", "feature", "sum(gsc_clicks) / sum(gsc_impressions), feature window"),
    ("pos_f", "feature", "sum(gsc_avg_position * gsc_impressions) / sum(gsc_impressions), feature window"),
    ("content_type", "feature", "dim_content.content_type"),
    ("age_days", "feature", "2026-03-16 - dim_content.content_created_date"),
    ("under_captures_next", "label", "outcome-window CTR < 0.5 x median CTR of its position bucket"),
    ("client_hash_id", "context", "client-holdout split only"),
    ("content_hash_id", "context", "join key, grain"),
], columns=["field", "bucket", "definition"])
print(fields.to_string(index=False))


              field  bucket                                                                     definition
             impr_f feature                            sum(gsc_impressions) where report_date < 2026-03-16
              ctr_f feature                         sum(gsc_clicks) / sum(gsc_impressions), feature window
              pos_f feature sum(gsc_avg_position * gsc_impressions) / sum(gsc_impressions), feature window
       content_type feature                                                       dim_content.content_type
           age_days feature                                  2026-03-16 - dim_content.content_created_date
under_captures_next   label                   outcome-window CTR < 0.5 x median CTR of its position bucket
     client_hash_id context                                                      client-holdout split only
    content_hash_id context                                                                join key, grain


## 3. Verify it with queries (grain, counts, missing values, windows)

Exactly three verification queries on the mid-panel month `month=2026-03`:

1. **Grain:** one row of the daily table really is one page on one day (no duplicates on `report_date × client × content`).
2. **Slice size and date span:** how many rows, pages, clients and days the March partition holds.
3. **Availability:** how many rows survive `gsc_data_available IS TRUE`, plus the rows of my actual lane slice (visible pages with impressions in both windows).


In [3]:
# Query 1 - GRAIN: any (report_date, client, content) appearing more than once?
dupes = con.sql("""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM fact_mar
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Query 1 - duplicate grain keys found: {len(dupes)}  ->  {'grain holds: one row = one page-day' if dupes.empty else 'GRAIN BROKEN'}")


Query 1 - duplicate grain keys found: 0  ->  grain holds: one row = one page-day


In [4]:
# Query 2 - COUNT + DATE SPAN of the March slice
span = con.sql("""
    SELECT COUNT(*)                        AS rows,
           COUNT(DISTINCT content_hash_id) AS pages,
           COUNT(DISTINCT client_hash_id)  AS clients,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT report_date)     AS days
    FROM fact_mar
""").df()
print("Query 2 - size and span of month=2026-03:")
print(span.to_string(index=False))


Query 2 - size and span of month=2026-03:
   rows  pages  clients  first_day   last_day  days
9841378 331437       55 2026-03-01 2026-03-31    31


In [5]:
# Query 3 - AVAILABILITY: rows with Search Console data (IS TRUE), and what survives into the lane slice
avail = con.sql(f"""
    SELECT COUNT(*)                                                     AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)            AS gsc_available_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                            AND gsc_impressions > 0)                     AS rows_with_impressions,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)            AS ga4_available_rows
    FROM fact_mar
""").df()
avail["gsc_share"] = (avail["gsc_available_rows"] / avail["all_rows"]).round(3)
print("Query 3 - availability in month=2026-03:")
print(avail.to_string(index=False))


Query 3 - availability in month=2026-03:
 all_rows  gsc_available_rows  rows_with_impressions  ga4_available_rows  gsc_share
  9841378             3611061                3611061              413966      0.367


**What the three queries show:**
1. **Grain holds:** 0 duplicate `report_date × client × content` keys, so one row really is one page on one day.
2. **Slice:** March 2026 has **9,841,378 rows** covering **331,437 pages** from **55 clients**, spanning **2026-03-01 → 2026-03-31** (all 31 days present).
3. **Availability:** only **3,611,061 rows (36.7%)** pass `gsc_data_available IS TRUE`. In this release that's exactly the rows that have impressions, so the other ~63% are page-days with no Search Console row, not page-days with zero search interest. GA4 is far thinner (413,966 rows), which is one more reason GA4 stays out of this lane's features for now.

### Five features, from the same month

I build the page-level frame with one query: feature window = 1–15 March, outcome window = 16–31 March, both filtered with `gsc_data_available IS TRUE`. The lane keeps pages with ≥100 impressions in **both** halves and a feature-window position between 1 and 20 (visible pages). The label compares each page's outcome-window CTR to the median of its position bucket (1–3, 4–10, 11–20), where the bucket comes from the **feature-window** position.

In [6]:
frame = con.sql(f"""
    WITH f AS (   -- feature window
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impr_f,
               SUM(gsc_clicks)      AS clicks_f,
               SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS pos_f
        FROM fact_mar
        WHERE gsc_data_available IS TRUE AND report_date < DATE '{DECISION}'
        GROUP BY 1, 2
    ), o AS (     -- outcome window (label only)
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impr_o,
               SUM(gsc_clicks)      AS clicks_o
        FROM fact_mar
        WHERE gsc_data_available IS TRUE AND report_date >= DATE '{DECISION}'
        GROUP BY 1
    )
    SELECT f.client_hash_id, f.content_hash_id,
           f.impr_f, 100.0 * f.clicks_f / f.impr_f AS ctr_f, f.pos_f,
           c.content_type,
           DATE_DIFF('day', c.content_created_date, DATE '{DECISION}') AS age_days,
           o.impr_o, 100.0 * o.clicks_o / o.impr_o AS ctr_o
    FROM f
    JOIN o USING (content_hash_id)
    LEFT JOIN dim_content c USING (content_hash_id)
    WHERE f.impr_f >= 100 AND o.impr_o >= 100 AND f.pos_f BETWEEN 1 AND 20
""").df()

frame["pos_bucket"] = pd.cut(frame["pos_f"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
frame["expected_ctr_o"] = frame.groupby("pos_bucket", observed=True)["ctr_o"].transform("median")
frame["under_captures_next"] = (frame["ctr_o"] < 0.5 * frame["expected_ctr_o"]).astype(int)
assert frame["content_hash_id"].is_unique
assert (frame["age_days"].dropna() >= 0).all()

FEATURES = ["impr_f", "ctr_f", "pos_f", "content_type", "age_days"]
print(f"Lane frame: {len(frame):,} pages from {frame['client_hash_id'].nunique()} clients | "
      f"label base rate {frame['under_captures_next'].mean():.3f}")
print("Missing values per feature:", frame[FEATURES].isna().sum().to_dict())
frame[FEATURES + ["under_captures_next"]].head(5).round(3)


Lane frame: 57,392 pages from 36 clients | label base rate 0.398
Missing values per feature: {'impr_f': 0, 'ctr_f': 0, 'pos_f': 0, 'content_type': 0, 'age_days': 0}


,impr_f,ctr_f,pos_f,content_type,age_days,under_captures_next
0,119.0,0.840,12.319,keyword article,160,0
1,283.0,0.000,12.463,keyword article,160,1
2,440.0,0.455,11.845,keyword article,160,0
3,1621.0,0.864,7.546,keyword article,160,0
4,1402.0,0.214,9.078,keyword article,160,0


### The trap: one label-derived column on purpose

Quick score: a small gradient-boosted model, trained on ~80% of **clients** and scored on the held-out ~20% (ROC-AUC and Precision@50). First with the five honest features, then with **`ctr_o`** added, the outcome-window CTR the label is computed from. It "knows the answer", so the score should jump towards perfect. Then I delete it and keep the honest number.

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

def prep(df, cols):
    X = df[cols].copy()
    X["impr_f"] = np.log1p(X["impr_f"])
    if "content_type" in X:
        X["content_type"] = X["content_type"].fillna("unknown").astype("category")
    return X

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

clients = frame["client_hash_id"].unique()
test_clients = set(np.random.default_rng(42).permutation(clients)[: max(1, round(len(clients) * 0.2))])
is_test = frame["client_hash_id"].isin(test_clients).to_numpy()
y = frame["under_captures_next"].to_numpy()

def quick_score(cols, name):
    X = prep(frame, cols)
    m = HistGradientBoostingClassifier(max_depth=3, max_iter=200, random_state=42, categorical_features="from_dtype")
    m.fit(X[~is_test], y[~is_test])
    s = m.predict_proba(X[is_test])[:, 1]
    print(f"{name:<38} ROC-AUC {roc_auc_score(y[is_test], s):.3f} | P@50 {precision_at_k(s, y[is_test]):.3f}")

print(f"Held-out: {is_test.sum():,} pages from {len(test_clients)} unseen clients, base rate {y[is_test].mean():.3f}\n")
quick_score(FEATURES + ["ctr_o"], "LEAKY  (5 features + ctr_o)")
quick_score(FEATURES, "HONEST (5 features)")


Held-out: 22,232 pages from 7 unseen clients, base rate 0.411



LEAKY  (5 features + ctr_o)            ROC-AUC 1.000 | P@50 1.000


HONEST (5 features)                    ROC-AUC 0.804 | P@50 0.820


**What happened:** with `ctr_o` added, the score jumps towards perfect. The model isn't learning anything about pages; it reads the answer back, because the label is literally "is `ctr_o` below half its bucket median". On the real decision day (16 March) that column doesn't exist yet. **So `ctr_o` is deleted from the feature list (below), and the honest number is the one I keep.** The same lesson as notebook 02's `trend_pct`, now on warehouse data.

In [8]:
# Delete the leak for good - only the five honest features remain
frame = frame.drop(columns=["ctr_o"])
assert "ctr_o" not in frame.columns and set(FEATURES) <= set(frame.columns)
print("Kept features:", FEATURES)
quick_score(FEATURES, "HONEST (kept)")


Kept features: ['impr_f', 'ctr_f', 'pos_f', 'content_type', 'age_days']


HONEST (kept)                          ROC-AUC 0.804 | P@50 0.820


## 4. Data limits

**Named limitation of my slice: it measures *one two-week hop inside one month*, on an unbalanced panel.**

- A 15-day outcome window is short: a page's CTR over two weeks is noisy, and some "under-capture" is just a quiet fortnight (weekday mix, a holiday, one news spike). The label is a proxy for "persistently under-clicking", not proof of it.
- Not every client has search data in March: clients' Search Console history starts on different dates (`dim_clients.gsc_data_start`), and only the rows with `gsc_data_available IS TRUE` count. A missing page is **not tracked**, which is different from **zero traffic**.
- Position is a daily average from Search Console; SERP features (answer boxes, AI overviews, ads) that also steal clicks aren't in the data, so a low CTR can't be pinned on the page itself.
- One month can't show seasonality, and March may not look like other months. Before trusting any number, the same contract must be re-run on other months, with June 2026 kept as the sealed final test.

The cell below shows how uneven the panel is going into March.


In [9]:
# How uneven is the panel? Clients with search data in March vs their history start
cov = con.sql("""
    SELECT c.client_hash_id,
           c.gsc_data_start,
           COUNT(DISTINCT f.report_date) FILTER (WHERE f.gsc_data_available IS TRUE) AS gsc_days_in_march
    FROM dim_clients c
    LEFT JOIN fact_mar f USING (client_hash_id)
    GROUP BY 1, 2
""").df()
print(f"Clients in dim_clients: {len(cov)} | with any GSC data in March: {(cov['gsc_days_in_march'] > 0).sum()} "
      f"| with all 31 days: {(cov['gsc_days_in_march'] == 31).sum()}")
print(f"Clients in the lane frame: {frame['client_hash_id'].nunique()}")
largest = frame["client_hash_id"].value_counts(normalize=True).iloc[0]
print(f"Largest single client's share of lane pages: {largest:.0%}")


Clients in dim_clients: 104 | with any GSC data in March: 47 | with all 31 days: 37
Clients in the lane frame: 36
Largest single client's share of lane pages: 26%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.